# Notebook 03 · Portugal vs Croacia + evento Cristiano Ronaldo

Este notebook amplía la simulación de partido con:

- prórroga;
- tanda de penaltis;
- probabilidad de gol de Cristiano;
- probabilidad de que Portugal avance y Cristiano marque;
- escenario extremo de fallo y eliminación.

El objetivo es mostrar cómo Monte Carlo permite estudiar eventos compuestos.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if (repo_root / "pmcw").exists():
    sys.path.insert(0, str(repo_root))

from pmcw import AccessoryEventConfig, RonaldoEventSimulation

## 1. Parámetros del escenario

In [ ]:
config = AccessoryEventConfig(
    n_simulations=100_000,
    seed=20260702,
    lambda_por_90=1.36,
    lambda_cro_90=1.02,
    fatigue_factor=0.85,
    ronaldo_goal_share=0.30,
    ronaldo_on_pitch_after_120=0.60,
    ronaldo_shootout_taker_probability=0.95,
    ronaldo_shootout_conversion=0.76,
)

config

## 2. Ejecución

In [ ]:
result = RonaldoEventSimulation(config).run().summary()

result_df = pd.DataFrame(
    [{"Evento":k, "Probabilidad":v} for k, v in result.items()]
)
result_df["Porcentaje"] = result_df["Probabilidad"] * 100
result_df

In [ ]:
ax = result_df.plot(
    x="Evento", y="Porcentaje", kind="bar",
    legend=False, figsize=(12,6)
)
ax.set_title("Probabilidades principales")
ax.set_ylabel("Probabilidad (%)")
ax.set_xlabel("")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 3. Jerarquía de eventos

No todos los eventos están al mismo nivel. El escenario extremo necesita que ocurran varias condiciones consecutivas:

1. El partido llega a penaltis.
2. Cristiano sigue disponible.
3. Cristiano lanza.
4. Cristiano falla.
5. Portugal queda eliminado.

In [ ]:
key_events = pd.DataFrame({
    "Evento":[
        "Portugal avanza",
        "Llega a penaltis",
        "Cristiano marca",
        "Portugal avanza + Cristiano marca",
        "Cristiano falla en tanda",
        "Escenario ocaso",
    ],
    "Probabilidad":[
        result["portugal_advances"],
        result["goes_to_penalties"],
        result["ronaldo_scores_before_shootout"],
        result["portugal_advances_and_ronaldo_scores"],
        result["ronaldo_misses_in_shootout"],
        result["ocaso_scenario"],
    ],
})
key_events["Porcentaje"] = key_events["Probabilidad"] * 100
key_events

In [ ]:
ax = key_events.plot(
    x="Evento", y="Porcentaje", kind="bar",
    legend=False, figsize=(11,5)
)
ax.set_title("Jerarquía de eventos compuestos")
ax.set_ylabel("Probabilidad (%)")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 4. Sensibilidad al peso de gol de Cristiano

`ronaldo_goal_share` controla la cuota probabilística de goles portugueses atribuible a Cristiano.

In [ ]:
rows = []
for goal_share in np.linspace(0.15, 0.45, 13):
    cfg = AccessoryEventConfig(
        n_simulations=40_000,
        seed=20260702,
        lambda_por_90=1.36,
        lambda_cro_90=1.02,
        fatigue_factor=0.85,
        ronaldo_goal_share=float(goal_share),
        ronaldo_on_pitch_after_120=0.60,
        ronaldo_shootout_taker_probability=0.95,
        ronaldo_shootout_conversion=0.76,
    )
    r = RonaldoEventSimulation(cfg).run().summary()
    rows.append({
        "ronaldo_goal_share":goal_share,
        "ronaldo_scores":r["ronaldo_scores_before_shootout"],
        "portugal_advances_and_ronaldo_scores":
            r["portugal_advances_and_ronaldo_scores"],
    })

goal_sensitivity = pd.DataFrame(rows)
goal_sensitivity.head()

In [ ]:
ax = goal_sensitivity.plot(
    x="ronaldo_goal_share",
    y=["ronaldo_scores","portugal_advances_and_ronaldo_scores"],
    figsize=(10,5)
)
ax.set_title("Sensibilidad al peso de gol de Cristiano")
ax.set_xlabel("ronaldo_goal_share")
ax.set_ylabel("Probabilidad")
plt.tight_layout()
plt.show()

## 5. Sensibilidad de la conversión en tanda

In [ ]:
rows = []
for conversion in np.linspace(0.60, 0.90, 13):
    cfg = AccessoryEventConfig(
        n_simulations=40_000,
        seed=20260702,
        lambda_por_90=1.36,
        lambda_cro_90=1.02,
        fatigue_factor=0.85,
        ronaldo_goal_share=0.30,
        ronaldo_on_pitch_after_120=0.60,
        ronaldo_shootout_taker_probability=0.95,
        ronaldo_shootout_conversion=float(conversion),
    )
    r = RonaldoEventSimulation(cfg).run().summary()
    rows.append({
        "conversion":conversion,
        "ronaldo_miss":r["ronaldo_misses_in_shootout"],
        "ocaso":r["ocaso_scenario"],
    })

penalty_sensitivity = pd.DataFrame(rows)
penalty_sensitivity.head()

In [ ]:
ax = penalty_sensitivity.plot(
    x="conversion",
    y=["ronaldo_miss","ocaso"],
    figsize=(10,5)
)
ax.set_title("Sensibilidad a la conversión en tanda")
ax.set_xlabel("Probabilidad de conversión")
ax.set_ylabel("Probabilidad")
plt.tight_layout()
plt.show()

## 6. Conclusiones

- Portugal aparece como favorito, pero no domina el cruce.
- El evento combinado tiene menor probabilidad que sus componentes.
- La tanda introduce más incertidumbre.
- El escenario extremo es residual.
- Los análisis de sensibilidad muestran qué resultados dependen más de los supuestos.